# 实验 00：运行环境与 GPU
目标：确认正在使用的 Python 内核，检查 CUDA，完成一次张量反向传播。
阅读：[D2L 5.6 GPU](https://zh.d2l.ai/chapter_deep-learning-computation/use-gpu.html)。
安装步骤见项目 README，安装完成后应重启并选择正确的内核。


In [ ]:
from pathlib import Path
import os, sys

# 从项目根目录、notebooks/ 或它们的子目录启动时都能找到 labkit。
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "labkit").is_dir()), None)
if ROOT is None:
    raise RuntimeError("请将整个课程目录解压后，从项目根目录启动 JupyterLab")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目根目录:", ROOT)


In [ ]:
from check_environment import main
main()


## 张量、损失与梯度
参数、输入必须处于相同设备。`backward()` 计算梯度，优化器的 `step()` 更新参数。


In [ ]:
import torch
from labkit.common import choose_device, seed_everything
seed_everything(42)
device = choose_device("auto")
model = torch.nn.Linear(4, 2).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
x = torch.randn(8, 4, device=device)
y = torch.tensor([0, 1] * 4, device=device)
before = model.weight.detach().clone()
optimizer.zero_grad(set_to_none=True)
logits = model(x)
loss = torch.nn.CrossEntropyLoss()(logits, y)
loss.backward()
optimizer.step()
print("device =", device, "loss =", loss.item())
print("参数确实更新:", not torch.equal(before, model.weight))


## 数据准备
在项目终端执行 `python prepare_data.py --dataset all`。全套实验只使用 CIFAR-10 和 IMDB。
教师也可以复制已经下载好的 `data/` 目录供学生离线使用。

## 检查题
1. `nvidia-smi` 能显示 GPU，但 `torch.cuda.is_available()` 为 False 时，你会检查什么？
2. `model.eval()` 和 `torch.no_grad()` 各自解决什么问题？
3. 修改随机种子，观察初始损失是否改变。
